In [ ]:
import torch
import matplotlib.pyplot as plt

SEED = 0

In [ ]:
torch.manual_seed(0)
weights = torch.tensor([0.5, 0.3, 0.2])
means = torch.tensor([-2.0, 0.5, 2.5])
stds = torch.tensor([0.3, 0.5, 0.2])

N = 500
k = torch.multinomial(weights, N, replacement=True)
x_1 = means[k] + stds[k] * torch.randn(N)

In [ ]:
x = torch.linspace(-4, 4, 1000)
dens = (weights * torch.exp(-0.5 * ((x[:, None] - means) / stds) ** 2)
        / (stds * (2 * torch.pi) ** 0.5)).sum(dim=1)

plt.hist(x_1, bins=60, density=True, alpha=0.5, label="samples")
plt.plot(x, dens, label="true density")
plt.legend()
plt.show()

In [ ]:
def empirical_zhat(x, t, data):
    logw = -(x[:, None] - t * data[None, :]) ** 2 / (2 * (1 - t) ** 2)
    w = torch.softmax(logw, dim=1)
    return w @ data

In [ ]:
def gmm_zhat(x, t, weights, means, stds):
    a = (1 - t) ** 2 + t ** 2 * stds ** 2
    logr = torch.log(weights) - 0.5 * torch.log(a) - (x[:, None] - t * means) ** 2 / (2 * a)
    r = torch.softmax(logr, dim=1)
    cond = means + t * stds ** 2 / a * (x[:, None] - t * means)
    return (r * cond).sum(dim=1)

In [ ]:
def velocity(zhat, x, t):
    return (zhat - x) / (1 - t)